# 01 · Exploratory data analysis — EMSCAD

**Goal:** understand what separates fraudulent from real postings before modelling, and confirm the
data decisions baked into the pipeline (dedupe before split, stratified split).

**Data:** produced by the DVC pipeline — run `uv run dvc pull` (or `uv run dvc repro`) first.
Pattern analysis uses the **train split only**, so nothing here peeks at validation/test data.

In [ ]:
import json

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from fjd.data.clean import dedupe_key
from fjd.data.schema import BINARY_COLUMNS, CATEGORICAL_COLUMNS, TARGET, TEXT_COLUMNS
from fjd.utils.config import PROJECT_ROOT, load_params
from fjd.utils.plotting import CLASS_COLORS, CLASS_LABELS, INK_SECONDARY, SERIES, set_style

set_style()
params = load_params()
DATA, REPORTS = PROJECT_ROOT / "data", PROJECT_ROOT / "reports"

raw = pd.read_parquet(DATA / "interim" / "ingested.parquet")
train = pd.read_parquet(DATA / "processed" / "train.parquet")
reports = {p.stem: json.loads(p.read_text()) for p in REPORTS.glob("*.json")}
print(f"raw rows: {len(raw):,}   train rows: {len(train):,}")

## 1. Class balance and duplicates

In [ ]:
clean, split = reports["clean"], reports["split"]
summary = pd.DataFrame(
    {
        "rows": [reports["data_validation"]["rows"], clean["rows_out"]]
        + [split[s]["rows"] for s in ("train", "val", "test")],
        "fraud_rate": [reports["data_validation"]["fraud_rate"], np.nan]
        + [split[s]["fraud_rate"] for s in ("train", "val", "test")],
    },
    index=["raw", "after dedupe", "train", "val", "test"],
)
print(
    f"duplicates removed: {clean['duplicates_removed']:,} "
    f"({clean['duplicates_removed'] / clean['rows_in']:.1%} of raw); "
    f"groups with conflicting labels: {clean['conflicting_label_groups']}"
)
summary.style.format({"rows": "{:,}", "fraud_rate": "{:.2%}"}, na_rep="—")

Why dedupe matters: the same posting text re-listed under several locations would otherwise
land in both train and test, and every model would look better than it really is. Which class do the
duplicates come from?

In [ ]:
key = dedupe_key(raw, params["clean"]["dedupe_on"])
has_dup = key.duplicated(keep=False)
has_dup.groupby(raw[TARGET]).mean().rename(index=CLASS_LABELS).to_frame(
    "share of rows that have a duplicate"
).style.format("{:.1%}")

## 2. Missing fields

How often each field is left empty, per class. Missing-field flags are cheap, interpretable
features.

In [ ]:
cols = TEXT_COLUMNS + CATEGORICAL_COLUMNS
miss = train.groupby(TARGET)[cols].agg(lambda s: s.isna().mean()).T.rename(columns=CLASS_LABELS)
miss["gap"] = miss["Fraud"] - miss["Real"]
miss = miss.sort_values("gap")

fig, ax = plt.subplots(figsize=(8, 5.5))
y = np.arange(len(miss))
h = 0.38
for i, (cls, label) in enumerate(CLASS_LABELS.items()):
    ax.barh(y + (i - 0.5) * (h + 0.02), miss[label], height=h, color=CLASS_COLORS[cls], label=label)
ax.set_yticks(y, miss.index)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("share of postings with the field missing")
ax.set_title("Missing fields by class (sorted by fraud − real gap)")
ax.legend(loc="center right")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()
miss.style.format("{:+.1%}", subset="gap").format("{:.1%}", subset=["Real", "Fraud"])

## 3. Binary flags and categorical fields — fraud rate by value

In [ ]:
base_rate = train[TARGET].mean()


def fraud_rate_by(col: pd.Series, min_count: int = 30, top: int = 12) -> pd.DataFrame:
    g = train.groupby(col.fillna("(missing)"))[TARGET].agg(["mean", "size"])
    g = g[g["size"] >= min_count].sort_values("mean", ascending=False).head(top)
    return g.rename(columns={"mean": "fraud_rate", "size": "n"})


def plot_rates(ax: plt.Axes, rates: pd.DataFrame, title: str) -> None:
    rates = rates.iloc[::-1]
    ax.barh(rates.index.astype(str), rates["fraud_rate"], color=SERIES[1], height=0.65)
    ax.axvline(base_rate, color=INK_SECONDARY, lw=1, ls="--")
    for yy, (rate, n) in enumerate(zip(rates["fraud_rate"], rates["n"], strict=True)):
        ax.text(rate, yy, f" {rate:.0%} (n={n:,})", va="center", fontsize=8, color=INK_SECONDARY)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.3)


fig, axes = plt.subplots(1, 3, figsize=(12, 2.6))
for ax, col in zip(axes, BINARY_COLUMNS, strict=True):
    plot_rates(ax, fraud_rate_by(train[col].map({0: "no", 1: "yes"})), col)
fig.suptitle(
    f"Fraud rate by flag (dashed line = overall {base_rate:.1%})",
    x=0.01,
    ha="left",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
country = train["location"].str.split(",").str[0].str.strip().rename("country")
fields = {
    c: train[c]
    for c in [
        "employment_type",
        "required_experience",
        "required_education",
        "function",
        "industry",
    ]
} | {"country": country}
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
for ax, (name, col) in zip(axes.flat, fields.items(), strict=True):
    plot_rates(ax, fraud_rate_by(col), name)
fig.suptitle(
    f"Fraud rate by category (n ≥ 30, highest 12; dashed line = overall {base_rate:.1%})",
    x=0.01,
    ha="left",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

Salary is rarely given, but whether it is given may matter:

In [ ]:
has_salary = train["salary_range"].notna().map({True: "salary given", False: "no salary"})
fraud_rate_by(has_salary).style.format({"fraud_rate": "{:.1%}", "n": "{:,}"})

## 4. Text length

In [ ]:
lengths = train[TEXT_COLUMNS].apply(lambda s: s.fillna("").str.split().str.len())
lengths["total"] = lengths.sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
bins = np.logspace(0, np.log10(lengths["total"].max() + 1), 40)
for ax, col in zip(axes, ["company_profile", "description", "total"], strict=True):
    for cls, label in CLASS_LABELS.items():
        vals = lengths.loc[train[TARGET] == cls, col] + 1
        ax.hist(
            vals,
            bins=bins,
            weights=np.full(len(vals), 1 / len(vals)),
            histtype="step",
            lw=2,
            color=CLASS_COLORS[cls],
            label=label,
        )
    ax.set_xscale("log")
    ax.set_title(f"{col} (words + 1, log scale)", fontsize=10)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
axes[0].set_ylabel("share of postings")
axes[0].legend()
fig.suptitle("Word-count distributions by class", x=0.01, ha="left", fontweight="bold")
plt.tight_layout()
plt.show()
lengths.groupby(train[TARGET]).median().rename(index=CLASS_LABELS)

## 5. Anonymization placeholders

EMSCAD replaced URLs, emails and phone numbers with tokens like `#URL_<hash>#`. The hash is noise
for a model, but *whether* a posting contains one may be informative.

In [ ]:
text = train[TEXT_COLUMNS].fillna("").agg(" ".join, axis=1)
placeholders = pd.DataFrame(
    {kind: text.str.contains(f"#{kind}_", regex=False) for kind in ["URL", "EMAIL", "PHONE"]}
)
placeholders.groupby(train[TARGET]).mean().rename(index=CLASS_LABELS).style.format("{:.1%}")

## 6. Words that separate the classes

Smoothed log-odds of each word appearing in a fraudulent vs real posting (train split, words in
≥ 20 postings, placeholders removed). A preview of what a linear TF-IDF model will pick up.

In [ ]:
clean_text = text.str.replace(r"#(URL|EMAIL|PHONE)_[^#]*#", " ", regex=True).str.lower()
vec = CountVectorizer(
    min_df=20, binary=True, stop_words="english", token_pattern=r"(?u)\b[a-z]{3,}\b"
)
X = vec.fit_transform(clean_text)
y_arr = train[TARGET].to_numpy()
fraud_counts = np.asarray(X[y_arr == 1].sum(axis=0)).ravel()
real_counts = np.asarray(X[y_arr == 0].sum(axis=0)).ravel()
n_fraud, n_real = (y_arr == 1).sum(), (y_arr == 0).sum()
log_odds = np.log((fraud_counts + 1) / (n_fraud - fraud_counts + 1)) - np.log(
    (real_counts + 1) / (n_real - real_counts + 1)
)
words = pd.Series(log_odds, index=vec.get_feature_names_out())
top = pd.concat([words.nsmallest(15), words.nlargest(15).iloc[::-1]])

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top.index, top.values, color=[CLASS_COLORS[int(v > 0)] for v in top.values], height=0.7)
ax.axvline(0, color=INK_SECONDARY, lw=1)
ax.set_xlabel("log-odds  (← real · fraud →)")
ax.set_title("Words most associated with each class")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

Many of the strongest fraud words are company names or text-scraping artifacts (words glued
together where HTML was stripped). They point at **repeated fraud campaigns**: the same scam template
posted many times with small edits, which exact-text dedupe does not catch.

## 7. Near-duplicate templates across splits

Exact duplicates are gone, but near-duplicates remain: under a plain random split, about **25%** of
val/test postings shared their description opening with a training posting. The pipeline therefore
uses a **group-aware split**: postings linked by a shared description opening
(`split.template_chars` letters) or company profile form one group (`src/fjd/data/groups.py`), and
whole groups go to a single split. The check below should show 0% overlap.
This is the only cell that touches val/test, and it looks at overlap only, not at patterns.

In [ ]:
val = pd.read_parquet(DATA / "processed" / "val.parquet")
test = pd.read_parquet(DATA / "processed" / "test.parquet")
held_out = pd.concat([val, test], ignore_index=True)


def template(d: pd.DataFrame) -> pd.Series:
    return d["description"].fillna("").str.lower().str.replace(r"[^a-z]", "", regex=True).str[:300]


print(
    f"groups shared between train and val/test: "
    f"{len(set(train['group_id']) & set(held_out['group_id']))}"
)
train_tpl, held_tpl = template(train), template(held_out)
rows = {}
for cls, label in CLASS_LABELS.items():
    m = (held_out[TARGET] == cls) & (held_tpl.str.len() > 20)
    seen = held_tpl[m].isin(set(train_tpl[train[TARGET] == cls]))
    rows[label] = {"val+test rows": int(m.sum()), "share with a train template": seen.mean()}
pd.DataFrame(rows).T.style.format(
    {"val+test rows": "{:,.0f}", "share with a train template": "{:.1%}"}
)

## 8. Takeaways for modelling

1. **Imbalance:** about 4.5% fraud after dedupe. Use PR-AUC, class weights, and a tuned decision
   threshold. Accuracy would be about 95.5% for a model that never flags anything.
2. **Dedupe was necessary and skewed toward fraud:** 22% of fraudulent rows had an exact duplicate
   (14% of real rows). Raw fraud rate 4.84% fell to 4.48% after removing 1,966 duplicates.
3. **Missing-field flags are strong:** `company_profile` is missing in 67% of fraud vs 17% of real
   postings. `required_experience`, `benefits` and `requirements` show smaller gaps. Salary is
   missing *less* often in fraud (scams advertise pay).
4. **Metadata is informative:** no company logo means 15% fraud vs 2% with a logo. Part-time,
   telecommuting, entry-level, Administrative/Accounting functions, Oil & Energy and AU/US locations
   are over-represented. Country is worth extracting from `location`.
5. **Placeholders carry signal:** `#EMAIL_…#` and `#PHONE_…#` tokens appear 2–4× more often in
   fraud. Replace them with a fixed token (keep the presence, drop the hash).
6. **Text length:** fraud is shorter overall, mostly because of the empty company profile.
   Description length alone separates the classes only weakly.
7. **Watch out for memorisation:** the top fraud words are company names and scraping artifacts
   from repeated campaigns, and about a quarter of val/test postings share a description template
   with train under a random split. That rewards recognising known templates and overstates
   performance on new scams, so the pipeline splits by template/company group instead. Expect lower
   but honest metrics, and **use `group_id` for every cross-validation** (e.g. `StratifiedGroupKFold`)
   during tuning, or the leak comes back through CV.